# CV Gap Analyser — Semantic Job Matching with HuggingFace and Pinecone

I built this tool while preparing my own job application. Comparing a CV against a job description is something every job seeker does manually — reading both documents, noting which skills appear in one but not the other, estimating alignment. This project automates that process using semantic embeddings rather than keyword matching. The distinction matters: a CV that describes 'deploying models to production cloud infrastructure' will match a job description asking for 'MLOps and cloud-native deployment' even though the exact words differ. Keyword matching misses this. Cosine similarity between sentence embeddings captures it.

In [ ]:
# Imports and config
import sys
sys.path.insert(0, '..')

from config import (
    EMBEDDING_MODEL, EMBEDDING_DIMENSION,
    STRONG_MATCH_THRESHOLD, GOOD_MATCH_THRESHOLD, PARTIAL_MATCH_THRESHOLD,
    PINECONE_INDEX_NAME, TOP_K_SIMILAR
)
from embeddings import embed_text, get_embed_model
from skill_extractor import extract_skills, compare_skills
from scorer import compute_match_score, compute_rouge_overlap, full_analysis
from url_fetcher import fetch_job_from_url

import numpy as np
print(f'Embedding model: {EMBEDDING_MODEL}')
print(f'Embedding dimension: {EMBEDDING_DIMENSION}')
print(f'Thresholds — Strong: {STRONG_MATCH_THRESHOLD}, Good: {GOOD_MATCH_THRESHOLD}, Partial: {PARTIAL_MATCH_THRESHOLD}')

## Semantic matching vs keyword matching

Keyword matching counts shared words. Semantic matching measures shared meaning. all-MiniLM-L6-v2 was pretrained on 1 billion sentence pairs to produce embeddings where semantically similar texts cluster in the same region of a 384-dimensional space. Two sentences about the same concept — even using completely different words — will have high cosine similarity.

This matters for job matching. A CV that describes experience with 'managed ML infrastructure on cloud platforms' will score high semantic similarity against a job requiring 'Amazon SageMaker and MLOps' even without those exact words.

In [ ]:
# Concrete semantic similarity examples
import numpy as np

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

phrases = [
    "Amazon SageMaker",
    "managed ML platform on AWS",
    "PostgreSQL database administration",
    "MLOps and cloud-native model deployment",
    "deploying models to production cloud infrastructure",
]

print('Loading embedding model...')
embeddings = {p: embed_text(p) for p in phrases}
print('Done.\n')

pairs = [
    ("Amazon SageMaker", "managed ML platform on AWS"),
    ("Amazon SageMaker", "PostgreSQL database administration"),
    ("MLOps and cloud-native model deployment", "deploying models to production cloud infrastructure"),
    ("Amazon SageMaker", "MLOps and cloud-native model deployment"),
]

print(f'{'Pair':<70} {'Similarity':>10}')
print('-' * 82)
for a, b in pairs:
    sim = cosine(embeddings[a], embeddings[b])
    pair_str = f'"{a}" vs "{b}"'
    print(f'{pair_str:<70} {sim:>10.4f}')

## URL extraction with trafilatura

Most job descriptions live on company career pages or job boards — not as text files. trafilatura is a Python library built for academic web crawling that identifies the main content area of a web page and extracts just that text, discarding navigation, footers, sidebars, and cookie banners. This makes it well-suited for job postings: every company career page is structured differently, and trafilatura handles most of them without custom rules.

The fallback to BeautifulSoup handles the cases where trafilatura returns too little content — typically JavaScript-rendered pages where the job description loads after the initial HTML. BeautifulSoup strips all HTML tags from the raw response and returns noisier but usable text.

The `/fetch-url` endpoint lets you verify extraction quality before running the full analysis.

In [ ]:
# URL extraction demonstration
# Using a publicly accessible job posting URL
# Replace with any real job posting URL to test extraction

DEMO_URL = "https://jobs.lever.co/openai"  # Replace with a real job posting URL

try:
    result = fetch_job_from_url(DEMO_URL)
    print(f"Extraction method: {result['extraction_method']}")
    print(f"Word count: {result['word_count']}")
    print(f"Truncated: {result['truncated']}")
    print(f"\nFirst 500 chars:")
    print(result['text'][:500])
except ValueError as e:
    print(f"Extraction failed (expected for demo URL): {e}")
    print("Replace DEMO_URL with a real job posting URL to test extraction.")

## Skill extraction with spaCy

Semantic similarity gives an overall match score. Skill extraction gives a specific, actionable gap list. The two-pass approach — keyword list matching plus spaCy noun phrase extraction — identifies which specific technologies and skills appear in the job description but not in the CV. The output answers a different question from the similarity score: not 'how well do these match overall' but 'specifically, what is missing'.

The keyword list covers ~80 common ML/data/software technologies. spaCy noun phrases catch technical terms that the keyword list misses — new tools, domain-specific jargon, or multi-word phrases like 'feature engineering' or 'batch transform'.

In [ ]:
# Skill extraction and comparison

SAMPLE_CV = """
Python developer with experience in machine learning and data science.
Skilled in TensorFlow, scikit-learn, pandas, and NumPy. Deployed models
to AWS Lambda and Azure App Service. Experience with Docker, CI/CD pipelines,
and REST APIs using FastAPI and Flask. Knowledge of NLP, deep learning,
and MLOps patterns including model versioning and experiment tracking with
MLflow. Built RAG pipelines using LangChain and ChromaDB. Amazon Bedrock
used across multiple projects for zero-shot classification benchmarking.
"""

SAMPLE_JD = """
We are looking for a Machine Learning Engineer with strong Python skills.
Requirements: TensorFlow or PyTorch, AWS or Azure cloud deployment, Docker,
FastAPI, MLflow, CI/CD, REST APIs. Experience with NLP and deep learning
preferred. Knowledge of MLOps practices and Amazon Bedrock required.
LangChain and RAG pipeline experience a plus. SageMaker for model deployment.
SHAP for model explainability. Kubernetes preferred.
"""

print('Extracting CV skills...')
cv_skills = extract_skills(SAMPLE_CV)
print(f'CV skills ({len(cv_skills)}): {cv_skills}\n')

print('Extracting JD skills...')
jd_skills = extract_skills(SAMPLE_JD)
print(f'JD skills ({len(jd_skills)}): {jd_skills}\n')

comparison = compare_skills(cv_skills, jd_skills)
print(f'Coverage: {comparison["coverage_label"]} ({comparison["match_rate"]*100:.0f}%)')
print(f'Matched: {comparison["matched"]}')
print(f'Missing: {comparison["missing"]}')
print(f'Additional (CV has, JD does not require): {comparison["additional"]}')

## ROUGE for ATS keyword alignment

ROUGE measures how much CV language appears verbatim in the job description. This matters for ATS (Applicant Tracking Systems) — automated screening tools that filter CVs based on keyword presence before a human reads them. A CV can score high on semantic similarity (same concepts, different words) but fail ATS screening because it uses different terminology. ROUGE flags this gap: the ideas are there but the keywords are not.

Recommendation: where ROUGE-L is low despite high semantic similarity, mirror the exact terminology from the job description in the CV.

In [ ]:
# ROUGE on three contrasting examples

examples = [
    {
        "label": "High semantic + high ROUGE (same concepts, same words)",
        "cv": "Experienced with TensorFlow, PyTorch, scikit-learn, Docker, FastAPI, MLflow, and AWS SageMaker.",
        "jd": "Requirements: TensorFlow, PyTorch, scikit-learn, Docker, FastAPI, MLflow, AWS SageMaker."
    },
    {
        "label": "High semantic + low ROUGE (same concepts, different words — ATS risk)",
        "cv": "Built production ML systems on cloud infrastructure with containerisation and automated deployment pipelines.",
        "jd": "Requirements: MLflow, Docker, Kubernetes, CI/CD, GitHub Actions, SageMaker."
    },
    {
        "label": "Low semantic + low ROUGE (genuinely poor match)",
        "cv": "Designed user interfaces in Figma. Built React components and CSS animations.",
        "jd": "Requirements: TensorFlow, PyTorch, MLflow, AWS SageMaker, Docker, Kubernetes."
    }
]

print(f'{'Example':<60} {'Semantic':>10} {'ROUGE-L':>10}')
print('-' * 82)
for ex in examples:
    sem = compute_match_score(ex['cv'], ex['jd'])
    rouge = compute_rouge_overlap(ex['cv'], ex['jd'])
    print(f'{ex["label"][:58]:<60} {sem["match_score_pct"]:>9}/100 {rouge["rougeL"]:>10.3f}')

## Pinecone job library

The built-in job library stores five realistic job descriptions as Pinecone vectors. When a CV is analysed, it is compared against all stored descriptions — returning the roles it matches most strongly. New job descriptions can be added via `POST /api/add-job` or `POST /api/add-job-url`. The URL route fetches and indexes a job posting directly from a career page — building a personal job library from real postings.

The index is seeded once at startup: the lifespan function checks the existing vector count and skips re-indexing if the library is already present. This avoids unnecessary upsert operations on every server restart.

In [ ]:
# Full analysis — realistic CV vs target role
from pathlib import Path

REALISTIC_CV = """
Xavier — AI / ML Engineer

SUMMARY
AI and Machine Learning Engineer specialised in LLM applications, MLOps, and
cloud-native ML deployment on AWS and Azure. Strong Python background across
the full data and ML stack.

EXPERIENCE

AI Engineer — Freelance (2024–present)
- Built LLM document summarizer using Amazon Bedrock (Claude) with LLMOps
  monitoring: token tracking, latency logging, cost dashboards.
- Benchmarked zero-shot classification across Bedrock models (Claude, Titan,
  Llama) on 1,000-document dataset. Compared accuracy, latency, cost.
- Deployed SageMaker MLOps pipeline: training job, model registry, batch
  transform, real-time endpoint with blue-green deployment via GitHub Actions.
- Built RAG pipeline using LangChain, HuggingFace sentence-transformers,
  and Pinecone vector database for semantic document retrieval.
- CV gap analyser: FastAPI REST API, semantic embeddings (all-MiniLM-L6-v2),
  Pinecone job library, ROUGE, spaCy skill extraction, trafilatura URL extraction.
- XGBoost credit risk model with SHAP explainability. Deployed to Azure App Service.
- LSTM stock trend predictor. TensorFlow, pandas, NumPy.

Data Analyst — Previous Role (2021–2023)
- Airflow ETL pipelines for daily data ingestion. Azure Data Factory.
- Power BI dashboards. dbt models for data transformation.
- PySpark jobs on Azure Databricks for large-scale log processing.

SKILLS
Python, SQL, pandas, NumPy, scikit-learn, XGBoost, TensorFlow, PyTorch
Amazon Bedrock, Amazon SageMaker, AWS Lambda, S3, EC2
Azure, Azure Data Factory, Azure ML, Azure App Service
Docker, GitHub Actions, CI/CD, Terraform
LangChain, RAG, HuggingFace, sentence-transformers, Pinecone, ChromaDB
FastAPI, REST API, pytest, MLflow, MLOps, LLMOps
Airflow, dbt, PySpark, Spark, Power BI, SHAP
"""

jd_path = Path('../job_library/data-science-mlops-engineer.txt')
if not jd_path.exists():
    jd_path = Path('job_library/data-science-mlops-engineer.txt')
TARGET_JD = jd_path.read_text(encoding='utf-8')

print('Running full analysis...')
result = full_analysis(
    cv_text=REALISTIC_CV,
    jd_text=TARGET_JD,
    jd_title='Data Science / MLOps Engineer',
    jd_company='Global Consulting Firm (Madrid)'
)

print('\n' + '='*60)
print('MATCH SCORE')
print('='*60)
print(f"Score: {result['match']['match_score_pct']}/100")
print(f"Label: {result['match']['match_label']}")
print(f"Similarity: {result['match']['similarity_score']:.4f}")

print('\n' + '='*60)
print('ROUGE SCORES')
print('='*60)
print(f"ROUGE-1: {result['rouge']['rouge1']:.4f}")
print(f"ROUGE-2: {result['rouge']['rouge2']:.4f}")
print(f"ROUGE-L: {result['rouge']['rougeL']:.4f}")
print(result['rouge']['interpretation'])

print('\n' + '='*60)
print('SKILL COVERAGE')
print('='*60)
skills = result['skills']
print(f"Coverage: {skills['coverage_label']} ({skills['match_rate']*100:.0f}%)")
print(f"Matched ({len(skills['matched'])}): {skills['matched']}")
print(f"Missing ({len(skills['missing'])}): {skills['missing']}")

print('\n' + '='*60)
print('SUMMARY')
print('='*60)
print(result['summary'])

print('\n' + '='*60)
print('RECOMMENDATIONS')
print('='*60)
for i, rec in enumerate(result['recommendations'][:5], 1):
    print(f'{i}. {rec}')

## Personal context

I used an early version of this tool while preparing my application for a Data Science / MLOps Engineer role at Accenture. The gap analysis identified Amazon Bedrock, SageMaker, and LLMOps as skills present in the job description but without strong portfolio evidence in my CV at the time. The projects built in response to those gaps — including the SageMaker MLOps pipeline, four Bedrock benchmarking projects, and the LLM document summarizer with LLMOps patterns — are now in the portfolio. The tool identified the gaps. The work closed them. This project is both a technical demonstration and a record of that process.

## Azure App Service Deployment

**Notes on dependencies at runtime:**

- **trafilatura**: makes outbound HTTP requests at inference time — fetching the job posting URL. Azure App Service allows outbound HTTP by default, so no additional configuration is needed. The `requests` library includes a User-Agent header to avoid being blocked by basic bot detection.
- **spaCy**: `en_core_web_sm` is downloaded at Docker build time, avoiding a slow runtime download on first request.
- **Pinecone**: API key stored as Azure App Service environment variable `PINECONE_API_KEY`. The application fails gracefully if the key is not set — all Pinecone operations return empty results rather than raising exceptions.

**Deployment commands:**

```bash
# Create resource group and app service plan
az group create --name cv-gap-analyser-rg --location westeurope
az appservice plan create --name cv-gap-analyser-plan --resource-group cv-gap-analyser-rg --sku F1 --is-linux
# Scale to F1 via portal after creation if needed

# Create web app
az webapp create --name cv-gap-analyser --resource-group cv-gap-analyser-rg --plan cv-gap-analyser-plan --runtime "PYTHON:3.11"

# Configure startup command
az webapp config set --name cv-gap-analyser --resource-group cv-gap-analyser-rg --startup-file "gunicorn main:app --workers 1 --worker-class uvicorn.workers.UvicornWorker --bind 0.0.0.0:8000 --timeout 600"

# Set environment variables
az webapp config appsettings set --name cv-gap-analyser --resource-group cv-gap-analyser-rg --settings SCM_DO_BUILD_DURING_DEPLOYMENT=true PINECONE_API_KEY=<your-key>

# Package and deploy
cd cv-gap-analyser && zip -r deploy.zip . -x "*.git*" -x "venv/*" -x "__pycache__/*" -x "*.ipynb_checkpoints*" -x "uploads/*"
az webapp deployment source config-zip --name cv-gap-analyser --resource-group cv-gap-analyser-rg --src deploy.zip
```

## Key Findings

TBD — populate after running the full analysis notebook top-to-bottom.

Include:
- Match score on sample CV vs data-science-mlops-engineer role
- Skill coverage rate
- Top missing skills identified
- ROUGE-L interpretation